# Expanding Window Forecasting — Model Horse Race

## What it does
Evaluates multiple ML models under **walk-forward expanding window** methodology:
at each time step `t`, each model is trained on all data up to `t` and makes a
one-step-ahead prediction for `t+1`. Hyperparameters are re-selected at each step
using **inner time-series cross-validation** — no look-ahead bias.

This replicates the most rigorous evaluation design used in the problem sets and the
academic finance literature (Campbell & Thompson 2008, Gu, Kelly & Xiu 2020).

## Models in the horse race
| Model | Key tuning parameter |
|---|---|
| LASSO | alpha (L1 penalty) |
| Ridge | alpha (L2 penalty) |
| ElasticNet | alpha + l1_ratio |
| PLS | n_components |
| PCR (PCA + Ridge) | n_components + alpha |

## Expanding vs rolling window
| Mode | Training set | Best when |
|---|---|---|
| **Expanding** | All past data (grows over time) | Stable DGP; more data = better |
| **Rolling** | Last `window_size` periods only | Structural breaks; older data hurts |

Set `WINDOW_TYPE = 'rolling'` and `ROLLING_WINDOW` to switch modes.

## Data format
A wide time-series matrix (rows = periods, columns = variables). The `TARGET_COL` column is
the series to forecast; all other numeric columns are predictors.
Example default: FREDMD.csv (monthly macro panel, ~130 variables).

## Configuration

In [ ]:
CONFIG = {
    # --- Data ---
    'DATA_FILE':        '../../data/FREDMD.csv',
    'DATE_COL':         'sasdate',
    'TARGET_COL':       'INDPRO',          # series to forecast
    # --- Cleaning ---
    'MIN_OBS_FRAC':     0.5,               # drop columns with too many NaNs
    # --- Expanding or rolling window ---
    'WINDOW_TYPE':      'expanding',       # 'expanding' or 'rolling'
    'ROLLING_WINDOW':   120,               # months; only used if WINDOW_TYPE='rolling'
    'INITIAL_TRAIN':    120,               # minimum months before first forecast
    # --- Inner CV for hyperparameter selection ---
    'N_SPLITS_INNER':   5,                 # TimeSeriesSplit folds
    # --- Hyperparameter grids ---
    'LASSO_ALPHAS':     [1e-4, 1e-3, 0.01, 0.1, 1.0],
    'RIDGE_ALPHAS':     [0.01, 0.1, 1.0, 10.0, 100.0],
    'ENET_ALPHAS':      [1e-3, 0.01, 0.1, 1.0],
    'ENET_L1_RATIOS':   [0.3, 0.5, 0.7],
    'PLS_N_COMPONENTS': [1, 2, 3, 5, 10],
    'PCR_N_COMPONENTS': [5, 10, 20, 30],
    'PCR_RIDGE_ALPHAS': [0.1, 1.0, 10.0],
    # --- Output ---
    'SAVE_RESULTS':     True,
    'OUTPUT_DIR':       'results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Data & Clean

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv, load_parquet, compute_oos_r2

if CONFIG['DATA_FILE'].endswith(('.pq', '.parquet')):
    df_raw = load_parquet(CONFIG['DATA_FILE'])
else:
    df_raw = load_csv(CONFIG['DATA_FILE'])

if CONFIG['DATE_COL'] in df_raw.columns:
    df_raw[CONFIG['DATE_COL']] = pd.to_datetime(df_raw[CONFIG['DATE_COL']])
    df_raw = df_raw.set_index(CONFIG['DATE_COL']).sort_index()

df_raw = df_raw.select_dtypes(include='number')
min_obs = int(len(df_raw) * CONFIG['MIN_OBS_FRAC'])
df_clean = df_raw.dropna(axis=1, thresh=min_obs)

assert CONFIG['TARGET_COL'] in df_clean.columns, \
    f"TARGET_COL '{CONFIG['TARGET_COL']}' not found."

print(f'Shape after cleaning : {df_clean.shape}')
print(f'Period               : {df_clean.index[0].date()} — {df_clean.index[-1].date()}')
print(f'Target               : {CONFIG["TARGET_COL"]}')
df_clean.head()

## Step 2 — Build Feature Matrix & Define Models

Each model is wrapped in a helper that:
1. Standardizes features (fit on training window only)
2. Runs inner `TimeSeriesSplit` CV to select hyperparameters
3. Refits on the full training window with the selected hyperparameters
4. Returns the one-step-ahead prediction

In [ ]:
predictor_cols = [c for c in df_clean.columns if c != CONFIG['TARGET_COL']]
X_all = df_clean[predictor_cols].values
y_all = df_clean[CONFIG['TARGET_COL']].values
dates = df_clean.index

print(f'Predictors : {len(predictor_cols)}')
print(f'Obs        : {len(y_all)}')

In [ ]:
def cv_score(model, X, y, n_splits):
    """Mean RMSE across TimeSeriesSplit folds."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr, va in tscv.split(X):
        m = type(model)(**model.get_params())
        m.fit(X[tr], y[tr])
        pred = m.predict(X[va])
        if pred.ndim > 1:
            pred = pred.ravel()
        scores.append(np.sqrt(mean_squared_error(y[va], pred)))
    return np.mean(scores)


def fit_predict(X_train, y_train, X_next):
    """Fit all models on X_train/y_train, predict X_next. Returns dict of predictions."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)
    Xn = scaler.transform(X_next)
    n  = CONFIG['N_SPLITS_INNER']
    preds = {}

    # --- LASSO ---
    best_a = min(CONFIG['LASSO_ALPHAS'],
                 key=lambda a: cv_score(Lasso(alpha=a, max_iter=5000), Xs, y_train, n))
    m = Lasso(alpha=best_a, max_iter=5000).fit(Xs, y_train)
    preds['LASSO'] = float(m.predict(Xn)[0])

    # --- Ridge ---
    best_a = min(CONFIG['RIDGE_ALPHAS'],
                 key=lambda a: cv_score(Ridge(alpha=a), Xs, y_train, n))
    m = Ridge(alpha=best_a).fit(Xs, y_train)
    preds['Ridge'] = float(m.predict(Xn)[0])

    # --- ElasticNet ---
    best_params = min(
        [(a, r) for a in CONFIG['ENET_ALPHAS'] for r in CONFIG['ENET_L1_RATIOS']],
        key=lambda p: cv_score(ElasticNet(alpha=p[0], l1_ratio=p[1], max_iter=5000),
                               Xs, y_train, n)
    )
    m = ElasticNet(alpha=best_params[0], l1_ratio=best_params[1], max_iter=5000).fit(Xs, y_train)
    preds['ElasticNet'] = float(m.predict(Xn)[0])

    # --- PLS ---
    max_pls = min(Xs.shape[1], Xs.shape[0] - 1)
    valid_pls = [k for k in CONFIG['PLS_N_COMPONENTS'] if k <= max_pls]
    best_k = min(valid_pls,
                 key=lambda k: cv_score(PLSRegression(n_components=k), Xs, y_train, n))
    m = PLSRegression(n_components=best_k).fit(Xs, y_train)
    preds['PLS'] = float(m.predict(Xn).ravel()[0])

    # --- PCR (PCA + Ridge) ---
    max_pc = min(Xs.shape[1], Xs.shape[0] - 1)
    valid_pc = [k for k in CONFIG['PCR_N_COMPONENTS'] if k <= max_pc]
    best_combo = min(
        [(k, a) for k in valid_pc for a in CONFIG['PCR_RIDGE_ALPHAS']],
        key=lambda p: _pcr_cv_score(Xs, y_train, p[0], p[1], n)
    )
    pca = PCA(n_components=best_combo[0]).fit(Xs)
    ridge = Ridge(alpha=best_combo[1]).fit(pca.transform(Xs), y_train)
    preds['PCR'] = float(ridge.predict(pca.transform(Xn))[0])

    return preds


def _pcr_cv_score(X, y, n_comp, alpha, n_splits):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr, va in tscv.split(X):
        pca = PCA(n_components=min(n_comp, len(tr) - 1, X.shape[1])).fit(X[tr])
        m   = Ridge(alpha=alpha).fit(pca.transform(X[tr]), y[tr])
        scores.append(np.sqrt(mean_squared_error(y[va], m.predict(pca.transform(X[va])))))
    return np.mean(scores)


print('Model helpers defined.')

## Step 3 — Walk-Forward Expanding (or Rolling) Window

This loop refit all models at every step. Depending on dataset size and grid resolution
it can take several minutes — reduce grid sizes in CONFIG to speed up.

In [ ]:
model_names  = ['LASSO', 'Ridge', 'ElasticNet', 'PLS', 'PCR']
all_preds    = {m: [] for m in model_names}
all_actuals  = []
all_dates    = []

T = len(y_all)
initial = CONFIG['INITIAL_TRAIN']

print(f'Walk-forward loop: {T - initial} steps')
print(f'Window type      : {CONFIG["WINDOW_TYPE"]}')
if CONFIG['WINDOW_TYPE'] == 'rolling':
    print(f'Rolling window   : {CONFIG["ROLLING_WINDOW"]} months')

for t in range(initial, T - 1):
    if CONFIG['WINDOW_TYPE'] == 'rolling':
        start = max(0, t - CONFIG['ROLLING_WINDOW'])
    else:
        start = 0

    X_tr = X_all[start:t]
    y_tr = y_all[start:t]
    X_next = X_all[t:t+1]

    # Drop NaN rows in training
    valid = ~np.isnan(y_tr) & ~np.any(np.isnan(X_tr), axis=1)
    X_tr, y_tr = X_tr[valid], y_tr[valid]

    # Impute NaNs in predictor matrix with column means
    col_means = np.nanmean(X_tr, axis=0)
    col_means = np.where(np.isnan(col_means), 0, col_means)
    X_tr   = np.where(np.isnan(X_tr),   col_means, X_tr)
    X_next = np.where(np.isnan(X_next), col_means, X_next)

    step_preds = fit_predict(X_tr, y_tr, X_next)

    for m in model_names:
        all_preds[m].append(step_preds[m])
    all_actuals.append(float(y_all[t + 1]))
    all_dates.append(dates[t + 1])

    if (t - initial + 1) % 12 == 0:
        print(f'  Step {t - initial + 1:4d}  ({dates[t+1].date()})  '
              f'Ridge pred={step_preds["Ridge"]:+.4f}  actual={all_actuals[-1]:+.4f}')

actuals = np.array(all_actuals)
print(f'\nCompleted {len(actuals)} forecasts.')

## Step 4 — OOS Metrics & Model Comparison

In [ ]:
# Historical mean benchmark (prevailing mean)
benchmark_mean = float(y_all[:initial].mean())

rows = []
for m in model_names:
    p      = np.array(all_preds[m])
    rmse   = float(np.sqrt(np.mean((actuals - p) ** 2)))
    mae    = float(np.mean(np.abs(actuals - p)))
    oos_r2 = float(compute_oos_r2(actuals, p, benchmark_mean))
    dir_acc = float(np.mean(np.sign(actuals) == np.sign(p)))
    rows.append({'Model': m, 'RMSE': rmse, 'MAE': mae,
                 'OOS R²': oos_r2, 'Dir. Acc.': dir_acc})

results_df = pd.DataFrame(rows).sort_values('OOS R²', ascending=False)

print('WALK-FORWARD EVALUATION — MODEL COMPARISON')
print('=' * 65)
print(f"  Window type : {CONFIG['WINDOW_TYPE']}")
print(f"  Forecasts   : {len(actuals)}")
print(f"  Target      : {CONFIG['TARGET_COL']}")
print()
print(results_df.to_string(index=False, float_format='{:.6f}'.format))
print('=' * 65)

## Step 5 — Forecast Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Actual vs forecasts
axes[0].plot(all_dates, actuals, color='black', linewidth=1.5, label='Actual', zorder=5)
colors = ['steelblue', 'tomato', 'seagreen', 'orange', 'purple']
for m, c in zip(model_names, colors):
    axes[0].plot(all_dates, all_preds[m], linewidth=1, alpha=0.7, label=m, color=c)
axes[0].set(ylabel=CONFIG['TARGET_COL'],
            title=f'{CONFIG["TARGET_COL"]} — Walk-Forward Forecasts ({CONFIG["WINDOW_TYPE"]} window)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# OOS R² bar chart
r2_vals  = results_df['OOS R²'].values
r2_names = results_df['Model'].values
bar_colors = ['steelblue' if v >= 0 else 'tomato' for v in r2_vals]
axes[1].barh(range(len(r2_vals)), r2_vals, color=bar_colors, alpha=0.8)
axes[1].set_yticks(range(len(r2_vals)))
axes[1].set_yticklabels(r2_names)
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set(xlabel='OOS R² (vs prevailing mean)', title='Model Comparison — OOS R²')
axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(r2_vals):
    axes[1].text(v + 0.0005, i, f'{v*100:+.4f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 6 — Cumulative OOS R² Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for m, c in zip(model_names, colors):
    p      = np.array(all_preds[m])
    ss_res = np.cumsum((actuals - p) ** 2)
    ss_tot = np.cumsum((actuals - benchmark_mean) ** 2)
    cum_r2 = 1 - ss_res / np.where(ss_tot == 0, 1e-12, ss_tot)
    ax.plot(all_dates, cum_r2 * 100, linewidth=1.5, label=m, color=c)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set(xlabel='Date', ylabel='Cumulative OOS R² (%)',
       title=f'Cumulative OOS R² — {CONFIG["TARGET_COL"]} ({CONFIG["WINDOW_TYPE"]} window)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    # Model comparison summary
    results_df['target']      = CONFIG['TARGET_COL']
    results_df['window_type'] = CONFIG['WINDOW_TYPE']
    results_df['n_forecasts'] = len(actuals)
    results_df['notebook']    = 'expanding_window'
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'expanding_window_summary.csv')
    results_df.to_csv(path, index=False)
    print(f'Saved: {path}')

    # Per-model forecast time series
    forecast_df = pd.DataFrame({'date': all_dates, 'actual': actuals})
    for m in model_names:
        forecast_df[f'pred_{m}'] = all_preds[m]
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'expanding_window_forecasts.csv')
    forecast_df.to_csv(path, index=False)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')